In [2]:
import site
from importlib import reload
reload(site)

import openai
print("✅ OpenAI versión cargada con éxito:", openai.__version__)

StatementMeta(, d439cac7-22cd-48b0-8001-3fbac567c714, 6, Finished, Available, Finished, False)

✅ OpenAI versión cargada con éxito: 3.3.1


In [3]:
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, DoubleType, TimestampType
)

# 1. Esquema gold_questions
schema_questions = StructType([
    StructField("question_id", StringType(), False),
    StructField("question_text", StringType(), False),
    StructField("question_embedding", ArrayType(DoubleType()), False),
    StructField("created_at", TimestampType(), False)
])

# 2. Esquema gold_answers
schema_answers = StructType([
    StructField("answer_id", StringType(), False),
    StructField("question_id", StringType(), False),
    StructField("answer_text", StringType(), False),
    StructField("retrieved_chunks", ArrayType(StringType()), False),
    StructField("llm_model", StringType(), False),
    StructField("created_at", TimestampType(), False)
])

# Crear tablas en Lakehouse si no existen
spark.createDataFrame([], schema_questions).write.format("delta").mode("ignore").saveAsTable("gold_questions")
spark.createDataFrame([], schema_answers).write.format("delta").mode("ignore").saveAsTable("gold_answers")

print("Tablas 'gold_questions' y 'gold_answers' verificadas en LH_Ecodocs.")

StatementMeta(, d439cac7-22cd-48b0-8001-3fbac567c714, 7, Finished, Available, Finished, False)

Tablas 'gold_questions' y 'gold_answers' verificadas en LH_Ecodocs.


In [ ]:
import os
import uuid
import time
import requests
import numpy as np
from datetime import datetime
from pyspark.sql import Row
from urllib3.util import Retry
from requests.adapters import HTTPAdapter
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# CONFIGURACIÓN DE CLIENTES Y MODELOS (GROQ API REST)
# ---------------------------------------------------------
GROQ_API_KEY = "TU_API_KEY_AQUI"
LLM_MODEL = "openai/gpt-oss-20b"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

# Sesión HTTP síncrona optimizada para el runtime de Fabric
http_session = requests.Session()
retries = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
http_session.mount("https://", HTTPAdapter(max_retries=retries))

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# ---------------------------------------------------------
# CARGA DEL ÍNDICE VECTORIAL EN MEMORIA
# ---------------------------------------------------------
df_chunks = spark.table("gold_document_chunks")
df_embeddings = spark.table("gold_embeddings")

df_gold_joined = df_chunks.join(df_embeddings, on=["chunk_id", "document_id"], how="inner").select(
    "chunk_id", "document_id", "document_title", "document_category", "source_url", "page_number", "chunk_text", "embedding"
)

gold_data_cache = df_gold_joined.collect()
embeddings_matrix = np.array([r.embedding for r in gold_data_cache], dtype=np.float32)
norms = np.linalg.norm(embeddings_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1e-10
normalized_embeddings_matrix = embeddings_matrix / norms

# ---------------------------------------------------------
# PROMPT BASE DE ECODOCS AI
# ---------------------------------------------------------
SYSTEM_PROMPT = """Eres EcoDocs AI, un asistente documental interno de EcoPower Solutions.
Responde únicamente usando el contexto proporcionado.
No inventes información.
Si el contexto no contiene información suficiente, responde:
"No tengo información suficiente en los documentos disponibles."
Incluye siempre las fuentes usadas al final."""

# ---------------------------------------------------------
# FUNCIÓN PRINCIPAL DE GENERACIÓN RAG
# ---------------------------------------------------------
def ask_ecodocs_ai(question_text: str, top_k: int = 2):
    """
    Ejecuta el flujo RAG completo:
    1. Vectoriza la pregunta y guarda en gold_questions.
    2. Recupera los chunks más relevantes.
    3. Construye el prompt con contexto y llama a Groq API vía REST síncrono.
    4. Guarda la respuesta y chunks en gold_answers.
    """
    if not question_text or not question_text.strip():
        print("La pregunta no puede estar vacía.")
        return None

    question_id = f"QST_{uuid.uuid4().hex[:8]}"
    answer_id = f"ANS_{uuid.uuid4().hex[:8]}"
    now = datetime.now()

    # 1. Vectorizar la pregunta
    query_vector = embedding_model.encode(question_text, normalize_embeddings=True)
    query_vector_list = [float(v) for v in query_vector]

    # Guardar en gold_questions
    q_row = Row(
        question_id=question_id,
        question_text=question_text,
        question_embedding=query_vector_list,
        created_at=now
    )
    df_q = spark.createDataFrame([q_row], schema=spark.table("gold_questions").schema)
    df_q.write.format("delta").mode("append").saveAsTable("gold_questions")

    # 2. Recuperación de Chunks Relevantes (Retrieval)
    query_vector_np = np.array(query_vector, dtype=np.float32)
    scores = np.dot(normalized_embeddings_matrix, query_vector_np)
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    retrieved_chunks_ids = []
    context_blocks = []
    sources = set()

    for idx in top_indices:
        rec = gold_data_cache[idx]
        retrieved_chunks_ids.append(rec.chunk_id)
        context_blocks.append(f"--- Documento: {rec.document_title} (Pág. {rec.page_number}) ---\n{rec.chunk_text}")
        sources.add(f"- {rec.document_title} (Pág. {rec.page_number})")

    retrieved_context_str = "\n\n".join(context_blocks)

    # 3. Construcción del User Prompt
    user_prompt = f"""Contexto:
{retrieved_context_str}

Pregunta:
{question_text}

Respuesta:"""

    # 4. Generación de Respuesta con llamadas HTTP REST puras
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.1
    }

    try:
        response = http_session.post(GROQ_URL, json=payload, headers=headers, timeout=30)
        
        # Manejo de Rate Limit manual (429) por si excede los reintentos automáticos
        if response.status_code == 429:
            print("⚠️ Rate limit alcanzado. Esperando 6 segundos...")
            time.sleep(6)
            response = http_session.post(GROQ_URL, json=payload, headers=headers, timeout=30)
            
        response.raise_for_status()
        answer_text = response.json()["choices"][0]["message"]["content"]
        
    except Exception as e:
        answer_text = f"Error al conectar con el LLM: {str(e)}"

    # 5. Persistir en gold_answers
    a_row = Row(
        answer_id=answer_id,
        question_id=question_id,
        answer_text=answer_text,
        retrieved_chunks=retrieved_chunks_ids,
        llm_model=LLM_MODEL,
        created_at=now
    )
    df_a = spark.createDataFrame([a_row], schema=spark.table("gold_answers").schema)
    df_a.write.format("delta").mode("append").saveAsTable("gold_answers")

    # Mostrar salida por pantalla
    print(f"\n Pregunta ID: {question_id}")
    print(f" Pregunta: {question_text}")
    print("=" * 80)
    print(f"🤖 Respuesta ({LLM_MODEL}):\n{answer_text}")
    print("=" * 80)

    return {"question_id": question_id, "answer_id": answer_id, "answer": answer_text}

StatementMeta(, d439cac7-22cd-48b0-8001-3fbac567c714, 8, Finished, Available, Finished, False)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

StatementMeta(, d439cac7-22cd-48b0-8001-3fbac567c714, 12, Finished, Available, Finished, False)

StatementMeta(, , -1, Finished, , Finished, True)

RejectSilentExecuteRequest: Livy session has failed. Error code: RejectSilentExecuteRequest. Rejected silent execute_request as there is no active session.

In [5]:
# PASO A: LOAD TEST QUESTIONS & EJECUCIÓN DEL BUCLE RAG
# ---------------------------------------------------------
test_questions = [
    "¿Cuántos días a la semana se permite teletrabajar en EcoPower?",
    "¿Entre qué horas aplica la franja protegida de desconexión digital?",
    "¿Cuál es el tiempo límite establecido para iniciar el mantenimiento correctivo ante una avería crítica en un aerogenerador?",
    "¿Con qué frecuencia deben realizarse las revisiones termográficas en las subestaciones eléctricas?",
    "¿Cuáles son los Equipos de Protección Individual (EPI) obligatorios para intervenir un inversor solar?",
    "¿A partir de qué velocidad de viento queda prohibido el ascenso a la torre de un aerogenerador?",
    "¿Cuál es el rango de tensión MPPT en el que deben operar los inversores fotovoltaicos?",
    "¿Cuál es la asignación presupuestaria máxima permitida para dietas diarias de manutención en viajes nacionales?",
    "¿Cuál es el valor económico máximo permitido para aceptar un regalo publicitario según el código ético?",
    "¿Qué sistema se utiliza para la limpieza sostenible de paneles solares fotovoltaicos?"
]

print(f"📄 Cargadas {len(test_questions)} preguntas de prueba. Iniciando procesamiento...\n")

# BUCLE DE EJECUCIÓN (Llama a ask_ecodocs_ai para cada pregunta)
for i, q in enumerate(test_questions, start=1):
    print(f"[{i}/10] Procesando pregunta...")
    ask_ecodocs_ai(q, top_k=2)
    time.sleep(3)  # Pausa para respetar la cuota de la API

print("\n¡Proceso finalizado con éxito! Las 10 preguntas y respuestas han sido registradas en las tablas Gold.")

StatementMeta(, d439cac7-22cd-48b0-8001-3fbac567c714, 9, Finished, Available, Finished, False)

📄 Cargadas 10 preguntas de prueba. Iniciando procesamiento...

[1/10] Procesando pregunta...

 Pregunta ID: QST_71990a78
 Pregunta: ¿Cuántos días a la semana se permite teletrabajar en EcoPower?
🤖 Respuesta (openai/gpt-oss-20b):
Se permite hasta **dos días a la semana** de teletrabajo.  

**Fuente:** Documento “Hr Politica Teletrabajo Ecopower” (pág. 1).
[2/10] Procesando pregunta...

 Pregunta ID: QST_f2d5eb66
 Pregunta: ¿Entre qué horas aplica la franja protegida de desconexión digital?
🤖 Respuesta (openai/gpt-oss-20b):
La franja protegida de desconexión digital se aplica **entre las 14:00 y las 15:00**.  

*Fuente: Política Interna de Teletrabajo de EcoPower Solutions (página 1).*
[3/10] Procesando pregunta...

 Pregunta ID: QST_42fa5c55
 Pregunta: ¿Cuál es el tiempo límite establecido para iniciar el mantenimiento correctivo ante una avería crítica en un aerogenerador?
🤖 Respuesta (openai/gpt-oss-20b):
No tengo información suficiente en los documentos disponibles.  

Fuentes consul